# Skin lesion screening: MobileNetV3-Large frozen backbone, low-data, offline Raspberry Pi pipeline
Demo framework notebook, simplified to a single backbone: **MobileNetV3-Large** (~3M backbone parameters), chosen because it is light
enough for real-time INT8 inference on a Raspberry Pi while still having enough capacity for subtle lesion features (colour, texture, border).
Larger or dermatology-specific foundation models may give richer features but are not realistic on-device targets.

**What this notebook does**
1. Loads a small dataset (default: capped at 800 training images, vs. ~8,000 typical of published melanoma classifiers).
2. Computes dataset-specific mean/std instead of generic ImageNet normalization.
3. Freezes the MobileNetV3-Large backbone entirely and trains only a small head on its 960-d embeddings.
4. Trains that head with a contrastive alignment loss between clean and simulated-degraded versions of the same lesion image.
5. Reports sensitivity retained vs. a clean baseline as a function of simulated image-quality drop (0-80%).
6. Exports an INT8-quantized ONNX model for the Pi and re-measures the same sensitivity-vs-quality curve on that quantized artifact.
7. Adds a Grad-CAM overlay (clean vs degraded) for the "why did it flag this" explanation.

**One-time install:** `pip install torch torchvision onnx onnxruntime onnxscript pandas scikit-learn matplotlib pillow`

**Dataset layout:** `dataset/train|val|test/<benign-folder>/*.jpg` and `dataset/train|val|test/melanoma/*.jpg` (split by lesion/patient, not by image).

*This is a scaffold, not a validated clinical tool.*

## 1. Config

In [ ]:
import os, io, json, copy, time, random, zlib
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageEnhance

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix, precision_score)


@dataclass
class Config:
    # data
    data_dir: Path = Path("dataset")        # use an absolute path if needed
    positive_class: str = "melanoma"        # name of the melanoma folder
    max_train_images: Optional[int] = 800   # the low-data regime (None = use everything)
    img_size: int = 224
    # model / head
    use_pretrained: bool = True             # ImageNet weights (internet needed the first time only)
    proj_dim: int = 128
    dropout: float = 0.3
    # training
    batch_size: int = 32
    epochs: int = 15
    lr: float = 1e-3
    contrastive_weight: float = 0.5
    temperature: float = 0.1
    train_severity: Tuple[int, int] = (20, 80)   # random quality drop (%) applied to the "degraded" view
    # evaluation
    eval_severities: Tuple[int, ...] = (0, 20, 40, 60, 80)
    target_sensitivity: float = 0.90
    # export
    n_calib: int = 200
    seed: int = 42
    out_dir: Path = Path("outputs")


CFG = Config()
CFG.out_dir.mkdir(exist_ok=True)

random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count() or 4)
print("Using device:", device)

## 2. Data

In [ ]:
def list_split(split):
    ds = datasets.ImageFolder(CFG.data_dir / split)
    pos = ds.class_to_idx[CFG.positive_class]          # melanoma = 1, everything else = 0
    return [p for p, _ in ds.samples], np.array([int(y == pos) for _, y in ds.samples])


def stratified_cap(paths, labels, n, seed=CFG.seed):
    if n is None or n >= len(paths):
        return paths, labels
    rng = np.random.RandomState(seed)
    keep = []
    for c in (0, 1):
        ci = np.where(labels == c)[0]
        keep.extend(rng.choice(ci, max(2, int(round(n * len(ci) / len(paths)))), replace=False))
    keep = sorted(int(i) for i in keep)
    return [paths[i] for i in keep], labels[keep]


train_paths, train_labels = stratified_cap(*list_split("train"), CFG.max_train_images)
val_paths, val_labels = list_split("val")
test_paths, test_labels = list_split("test")

for name, y in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    print(f"{name:5s}: {len(y):5d} images | melanoma {int(y.sum())} ({y.mean():.1%}) | benign {int((1 - y).sum())}")

## 3. Degradation model
One 0-100 "quality drop" knob combining low light, downsampling, blur, sensor noise and JPEG compression.

In [ ]:
def load_rgb(path):
    return Image.open(path).convert("RGB").resize((CFG.img_size, CFG.img_size), Image.BILINEAR)


def _lerp(severity, lo, hi):
    return lo + (hi - lo) * severity / 100.0


def degrade_image(img, severity, seed=None):
    """severity 0 = untouched, 100 = very poor phone photo."""
    if severity <= 0:
        return img
    rng = np.random.default_rng(seed)
    out = ImageEnhance.Brightness(img).enhance(_lerp(severity, 1.0, 0.5))                 # low light
    w, h = out.size
    scale = _lerp(severity, 1.0, 0.35)                                                     # low resolution
    out = out.resize((max(8, int(w * scale)), max(8, int(h * scale))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    sigma = _lerp(severity, 0.0, 3.0)                                                      # blur
    if sigma > 0.1:
        out = out.filter(ImageFilter.GaussianBlur(sigma))
    arr = np.asarray(out, dtype=np.float32)                                                # sensor noise
    arr = np.clip(arr + rng.normal(0, _lerp(severity, 0.0, 18.0), arr.shape), 0, 255).astype(np.uint8)
    buf = io.BytesIO()                                                                     # JPEG compression
    Image.fromarray(arr).save(buf, format="JPEG", quality=int(_lerp(severity, 95, 10)))
    buf.seek(0)
    return Image.open(buf).convert("RGB")


# preview
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
base = load_rgb(val_paths[0])
for ax, s in zip(axes, CFG.eval_severities):
    ax.imshow(degrade_image(base, s, seed=0)); ax.set_title(f"quality drop {s}%"); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Frozen MobileNetV3-Large backbone

In [ ]:
class MobileNetV3Backbone(nn.Module):
    """ImageNet MobileNetV3-Large without its classifier. Fully frozen; always in eval mode."""
    def __init__(self, pretrained=True):
        super().__init__()
        m = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT if pretrained else None)
        self.features, self.avgpool = m.features, m.avgpool
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    def train(self, mode=True):          # never leave eval mode (keeps BatchNorm statistics fixed)
        return super().train(False)

    def forward(self, x):                # (B,3,224,224) -> (B,960) embedding
        return torch.flatten(self.avgpool(self.features(x)), 1)


backbone = MobileNetV3Backbone(CFG.use_pretrained).to(device)
print(f"Frozen backbone parameters: {sum(p.numel() for p in backbone.parameters()):,}")

## 5. Own-data mean/std

In [ ]:
def compute_mean_std(paths, n=200, seed=CFG.seed):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(paths), min(n, len(paths)), replace=False)
    s, sq, cnt = np.zeros(3), np.zeros(3), 0
    for i in idx:
        a = np.asarray(load_rgb(paths[int(i)]), dtype=np.float64).reshape(-1, 3) / 255.0
        s += a.sum(0); sq += (a ** 2).sum(0); cnt += len(a)
    mean = s / cnt
    return mean.tolist(), np.sqrt(sq / cnt - mean ** 2).tolist()


MEAN, STD = compute_mean_std(train_paths)
print("dataset mean:", np.round(MEAN, 3), "| std:", np.round(STD, 3))


def to_tensor(img):
    return TF.normalize(TF.to_tensor(img), MEAN, STD)


def load_tensor(path, severity):
    """Deterministic (repeatable) degraded view of one image."""
    return to_tensor(degrade_image(load_rgb(path), severity, seed=zlib.crc32(path.encode()) + int(severity)))


_geom_aug = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                                transforms.RandomRotation(180)])


class PairedTrainDataset(Dataset):
    """Returns (clean view, degraded view of the SAME augmented image, label)."""
    def __init__(self, paths, labels):
        self.paths, self.labels = paths, labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = _geom_aug(load_rgb(self.paths[i]))
        severity = random.uniform(*CFG.train_severity)
        return to_tensor(img), to_tensor(degrade_image(img, severity)), float(self.labels[i])


class EvalDataset(Dataset):
    def __init__(self, paths, labels, severity):
        self.paths, self.labels, self.severity = paths, labels, severity

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return load_tensor(self.paths[i], self.severity), int(self.labels[i])


@torch.no_grad()
def embed_paths(paths, labels, severity):
    dl = DataLoader(EvalDataset(paths, labels, severity), batch_size=CFG.batch_size, shuffle=False, num_workers=0)
    embs, ys = [], []
    for x, y in dl:
        embs.append(backbone(x.to(device)).cpu()); ys.append(y)
    return torch.cat(embs), torch.cat(ys).numpy()

## 6. Lightweight trainable head and contrastive alignment
The head is linear (projection -> classifier), so Grad-CAM reduces exactly to a gradient-free CAM that the Pi can compute.

In [ ]:
class ContrastiveHead(nn.Module):
    def __init__(self, in_dim=960, proj_dim=CFG.proj_dim, p=CFG.dropout):
        super().__init__()
        self.proj = nn.Linear(in_dim, proj_dim)     # embedding used for the contrastive alignment
        self.drop = nn.Dropout(p)
        self.cls = nn.Linear(proj_dim, 1)           # melanoma logit

    def forward(self, emb):
        z = self.proj(emb)
        return self.cls(self.drop(z)).squeeze(1), z

    def effective_weights(self):
        """Because proj and cls are both linear: logit = v . embedding + c."""
        W1, b1 = self.proj.weight.detach(), self.proj.bias.detach()
        w2, b2 = self.cls.weight.detach(), self.cls.bias.detach()
        return (w2 @ W1)[0].cpu().numpy(), float((w2 @ b1 + b2)[0])


def info_nce_loss(z_clean, z_deg, temperature=CFG.temperature):
    """Symmetric InfoNCE: a clean view should match its own degraded view, not other lesions in the batch."""
    a, b = F.normalize(z_clean, dim=1), F.normalize(z_deg, dim=1)
    logits = a @ b.T / temperature
    target = torch.arange(len(a), device=a.device)
    return 0.5 * (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target))


head = ContrastiveHead().to(device)
print(f"Trainable head parameters: {sum(p.numel() for p in head.parameters()):,}")

## 7. Training

In [ ]:
def train_head(head, epochs=CFG.epochs):
    train_dl = DataLoader(PairedTrainDataset(train_paths, train_labels), batch_size=CFG.batch_size,
                          shuffle=True, num_workers=0, drop_last=True)
    pos_weight = torch.tensor((len(train_labels) - train_labels.sum()) / max(train_labels.sum(), 1),
                              dtype=torch.float32).to(device)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(head.parameters(), lr=CFG.lr, weight_decay=1e-4)

    # validation embeddings are fixed (the backbone is frozen), so compute them once: clean + moderately degraded
    val_cache = {s: embed_paths(val_paths, val_labels, s) for s in (0, 50)}

    best_auc, best_state, hist = -1.0, None, []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        head.train()
        tot = {"cls": 0.0, "con": 0.0}
        for x_c, x_d, y in train_dl:
            y = y.to(device)
            with torch.no_grad():
                e_c, e_d = backbone(x_c.to(device)), backbone(x_d.to(device))
            logit_c, z_c = head(e_c)
            logit_d, z_d = head(e_d)
            cls_loss = bce(logit_c, y) + bce(logit_d, y)
            con_loss = info_nce_loss(z_c, z_d)
            loss = cls_loss + CFG.contrastive_weight * con_loss
            opt.zero_grad(); loss.backward(); opt.step()
            tot["cls"] += cls_loss.item(); tot["con"] += con_loss.item()

        head.eval()
        with torch.no_grad():
            aucs = [roc_auc_score(y, torch.sigmoid(head(e.to(device))[0]).cpu().numpy())
                    for e, y in val_cache.values()]
        val_auc = float(np.mean(aucs))
        n = len(train_dl)
        hist.append(dict(epoch=epoch, cls_loss=tot["cls"] / n, contrastive_loss=tot["con"] / n, val_auc=val_auc))
        print(f"epoch {epoch}/{epochs} | cls {tot['cls'] / n:.3f} | contrastive {tot['con'] / n:.3f} | "
              f"val AUC (clean/50%-degraded avg) {val_auc:.3f} | {time.time() - t0:.0f}s")
        if val_auc > best_auc:
            best_auc, best_state = val_auc, copy.deepcopy(head.state_dict())
    head.load_state_dict(best_state)
    return pd.DataFrame(hist)


hist_df = train_head(head)
torch.save({"head": head.state_dict(), "mean": MEAN, "std": STD}, CFG.out_dir / "head.pt")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(hist_df.cls_loss, label="classification"); ax[0].plot(hist_df.contrastive_loss, label="contrastive")
ax[0].legend(); ax[0].set_title("Training losses"); ax[0].set_xlabel("epoch")
ax[1].plot(hist_df.val_auc); ax[1].set_title("Validation AUC"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 8. Evaluation and decision threshold

In [ ]:
def pick_threshold(y, p, target_sens=CFG.target_sensitivity):
    """Highest threshold that still reaches the target sensitivity (fewest false alarms)."""
    _, tpr, thr = roc_curve(y, p)
    return float(thr[np.where(tpr >= target_sens)[0][0]])


def compute_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"roc_auc": roc_auc_score(y, p),
            "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1),
            "precision": precision_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "accuracy": accuracy_score(y, pred)}


@torch.no_grad()
def pytorch_scores(paths, labels, severity):
    emb, y = embed_paths(paths, labels, severity)
    head.eval()
    return y, torch.sigmoid(head(emb.to(device))[0]).cpu().numpy()


# threshold is chosen ONCE on clean validation images and then applied unchanged at every quality level
y_val, p_val = pytorch_scores(val_paths, val_labels, 0)
THRESHOLD = pick_threshold(y_val, p_val)
print(f"Threshold (clean validation, sensitivity >= {CFG.target_sensitivity:.0%}): {THRESHOLD:.4f}")

y_t, p_t = pytorch_scores(test_paths, test_labels, 0)
print(pd.DataFrame({"threshold 0.5": compute_metrics(y_t, p_t, 0.5),
                    f"threshold {THRESHOLD:.3f}": compute_metrics(y_t, p_t, THRESHOLD)}).round(4))

## 9. Sensitivity retention vs. simulated quality drop (PyTorch model)

In [ ]:
def quality_curve(score_fn, label):
    rows = []
    for s in CFG.eval_severities:
        y, p = score_fn(s)
        rows.append({"quality_drop_%": s, **compute_metrics(y, p, THRESHOLD)})
    df = pd.DataFrame(rows)
    base = max(df.sensitivity.iloc[0], 1e-9)
    df["sensitivity_retained_%"] = 100 * df.sensitivity / base
    df.insert(0, "model", label)
    return df


curve_pt = quality_curve(lambda s: pytorch_scores(test_paths, test_labels, s), "PyTorch FP32")
curve_pt.round(4).to_csv(CFG.out_dir / "quality_curve_pytorch.csv", index=False)
print(curve_pt.round(3).to_string(index=False))

## 10. Grad-CAM (clean vs degraded)

In [ ]:
class FullModel(nn.Module):
    """Backbone + head as one graph. Returns (logit, last 7x7x960 feature map)."""
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone, self.head = backbone, head

    def forward(self, x):
        f = self.backbone.features(x)
        pooled = torch.flatten(self.backbone.avgpool(f), 1)
        logit, _ = self.head(pooled)
        return logit.unsqueeze(1), f


full = FullModel(backbone, head).to(device).eval()


def compute_gradcam(model, x):
    """x: normalized tensor (1,3,224,224)."""
    model.eval()
    store = {}

    def fwd_hook(module, inp, out):
        store["acts"] = out.detach()
        out.register_hook(lambda g: store.__setitem__("grads", g.detach()))

    handle = model.backbone.features[-1].register_forward_hook(fwd_hook)
    try:
        x = x.clone().to(device).requires_grad_(True)       # guarantees a graph although everything is frozen
        with torch.enable_grad():
            logit = model(x)[0][0, 0]
            model.zero_grad()
            logit.backward()
    finally:
        handle.remove()

    acts, grads = store["acts"][0], store["grads"][0]
    cam = F.relu((grads.mean(dim=(1, 2))[:, None, None] * acts).sum(0))      # 7 x 7
    cam224 = F.interpolate(cam[None, None], size=(224, 224), mode="bilinear", align_corners=False)[0, 0]
    cam224 = (cam224 - cam224.min()) / (cam224.max() - cam224.min() + 1e-8)
    return cam224.cpu().numpy(), cam.cpu().numpy(), torch.sigmoid(logit).item(), acts


def denorm(t):
    return (t.cpu() * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1).permute(1, 2, 0).numpy()


def show_gradcam_pairs(indices, severity=60, save_path=None):
    """Top row: clean image + heatmap. Bottom row: the same lesion degraded + heatmap."""
    fig, axes = plt.subplots(4, len(indices), figsize=(2.8 * len(indices), 11), squeeze=False)
    for j, i in enumerate(indices):
        for r, sev in enumerate([0, severity]):
            x = load_tensor(test_paths[i], sev)
            cam, _, prob, _ = compute_gradcam(full, x.unsqueeze(0))
            img = denorm(x)
            axes[2 * r, j].imshow(img)
            axes[2 * r, j].set_title(f"{'clean' if sev == 0 else f'{sev}% drop'} | true: {'MEL' if test_labels[i] else 'benign'}", fontsize=8)
            axes[2 * r + 1, j].imshow(img); axes[2 * r + 1, j].imshow(cam, cmap="jet", alpha=0.45)
            axes[2 * r + 1, j].set_title(f"risk {prob:.2f} -> {'FLAG' if prob >= THRESHOLD else 'low'}", fontsize=8)
            axes[2 * r, j].axis("off"); axes[2 * r + 1, j].axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=140)
    plt.show()


pos, neg = np.where(test_labels == 1)[0], np.where(test_labels == 0)[0]
hits = pos[np.argsort(-p_t[pos])][:2]
misses = pos[np.argsort(p_t[pos])][:2]
false_alarms = neg[np.argsort(-p_t[neg])][:1]
chosen = [int(i) for i in np.concatenate([hits, misses, false_alarms])]
show_gradcam_pairs(chosen, severity=60, save_path=CFG.out_dir / "gradcam_clean_vs_degraded.png")

# Sanity check: both layers are linear, so Grad-CAM must equal CAM = ReLU(sum_k v_k * A_k) (up to scale)
v, c = head.effective_weights()
_, cam_raw, _, acts = compute_gradcam(full, load_tensor(test_paths[chosen[0]], 0).unsqueeze(0))
cam_head = F.relu((torch.tensor(v, device=acts.device)[:, None, None] * acts).sum(0)).cpu().numpy()
cos = float((cam_raw * cam_head).sum() / (np.linalg.norm(cam_raw) * np.linalg.norm(cam_head) + 1e-12))
print(f"Grad-CAM vs head-weight CAM cosine similarity: {cos:.4f} (should be ~1.0)")

## 11. Export to ONNX and INT8 quantization

In [ ]:
model_cpu = copy.deepcopy(full).cpu().eval()
onnx_fp32 = CFG.out_dir / "mnv3l_fp32.onnx"
export_kwargs = dict(input_names=["image"], output_names=["logit", "features"],
                     dynamic_axes={"image": {0: "batch"}, "logit": {0: "batch"}, "features": {0: "batch"}},
                     opset_version=17)
dummy = torch.randn(1, 3, CFG.img_size, CFG.img_size)
try:
    torch.onnx.export(model_cpu, dummy, str(onnx_fp32), dynamo=False, **export_kwargs)
except Exception as e:
    print("Legacy exporter unavailable, using default exporter:", repr(e)[:120])
    torch.onnx.export(model_cpu, dummy, str(onnx_fp32), **export_kwargs)

# files the Pi needs besides the model: heatmap weights + preprocessing + threshold
np.savez(CFG.out_dir / "head.npz", w=v, b=c)
json.dump({"mean": MEAN, "std": STD, "threshold": THRESHOLD}, open(CFG.out_dir / "config.json", "w"))

import onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat

xb = torch.stack([load_tensor(p, 0) for p in test_paths[:8]])
with torch.no_grad():
    t_logit = model_cpu(xb)[0].numpy()
o_logit = ort.InferenceSession(str(onnx_fp32), providers=["CPUExecutionProvider"]).run(None, {"image": xb.numpy()})[0]
print(f"FP32 ONNX size: {onnx_fp32.stat().st_size / 1e6:.2f} MB | max |PyTorch - ONNX| logit diff: {np.abs(t_logit - o_logit).max():.2e}")


class CalibReader(CalibrationDataReader):
    """Calibrates on a MIX of clean and degraded training images so INT8 ranges cover real field conditions."""
    def __init__(self, n=CFG.n_calib):
        rng = np.random.RandomState(0)
        idx = rng.choice(len(train_paths), min(n, len(train_paths)), replace=False)
        sevs = rng.choice([0, 0, 30, 60], len(idx))
        self.data = iter([{"image": load_tensor(train_paths[int(i)], int(s)).unsqueeze(0).numpy()}
                          for i, s in zip(idx, sevs)])

    def get_next(self):
        return next(self.data, None)


onnx_int8 = CFG.out_dir / "mnv3l_int8.onnx"
src = onnx_fp32
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
    quant_pre_process(str(onnx_fp32), str(CFG.out_dir / "mnv3l_fp32_prep.onnx"))
    src = CFG.out_dir / "mnv3l_fp32_prep.onnx"
except Exception as e:
    print("Skipping pre-processing step:", repr(e)[:100])

# QDQ with signed INT8 is the recommended setting for ARM CPUs (Raspberry Pi)
quantize_static(str(src), str(onnx_int8), CalibReader(), quant_format=QuantFormat.QDQ, per_channel=True,
                weight_type=QuantType.QInt8, activation_type=QuantType.QInt8)
print(f"INT8 ONNX size: {onnx_int8.stat().st_size / 1e6:.2f} MB")

## 12. Sensitivity vs. quality drop, re-measured on the INT8 artifact

In [ ]:
def onnx_scores_fn(path):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])

    def fn(severity):
        dl = DataLoader(EvalDataset(test_paths, test_labels, severity), batch_size=CFG.batch_size, shuffle=False, num_workers=0)
        ys, ps = [], []
        for x, y in dl:
            logit = sess.run(None, {"image": x.numpy()})[0]
            ps.append(1 / (1 + np.exp(-logit[:, 0]))); ys.append(y.numpy())
        return np.concatenate(ys), np.concatenate(ps)
    return fn


def latency_ms(path, runs=30):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    x = np.random.randn(1, 3, CFG.img_size, CFG.img_size).astype(np.float32)
    sess.run(None, {"image": x})
    t0 = time.time()
    for _ in range(runs):
        sess.run(None, {"image": x})
    return (time.time() - t0) / runs * 1000


curve_int8 = quality_curve(onnx_scores_fn(onnx_int8), "ONNX INT8")
curves = pd.concat([curve_pt, curve_int8])
curves.round(4).to_csv(CFG.out_dir / "quality_curve_pytorch_vs_int8.csv", index=False)
print(curves[["model", "quality_drop_%", "roc_auc", "sensitivity", "specificity", "sensitivity_retained_%"]].round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for label, df in curves.groupby("model", sort=False):
    ax[0].plot(df["quality_drop_%"], df["sensitivity_retained_%"], "o-", label=label)
    ax[1].plot(df["quality_drop_%"], df["roc_auc"], "o-", label=label)
ax[0].set_title("Sensitivity retained vs. clean (%)"); ax[1].set_title("ROC-AUC")
for a in ax:
    a.set_xlabel("simulated quality drop (%)"); a.legend()
plt.tight_layout(); plt.savefig(CFG.out_dir / "quality_curve.png", dpi=140); plt.show()

size_pt = (sum(p.numel() for p in backbone.parameters()) + sum(p.numel() for p in head.parameters())) * 4 / 1e6
summary = pd.DataFrame([
    {"model": "PyTorch FP32", "size_MB": round(size_pt, 2), "ms_per_image_this_PC": np.nan},
    {"model": "ONNX FP32", "size_MB": round(onnx_fp32.stat().st_size / 1e6, 2), "ms_per_image_this_PC": round(latency_ms(onnx_fp32), 1)},
    {"model": "ONNX INT8", "size_MB": round(onnx_int8.stat().st_size / 1e6, 2), "ms_per_image_this_PC": round(latency_ms(onnx_int8), 1)},
])
print(summary.to_string(index=False))
print("\nNote: INT8 can look slower than FP32 on a PC. The settings target ARM CPUs, so judge speed on the Pi itself.")

## 13. Raspberry Pi inference script

In [ ]:
PI_SCRIPT = r'''# Offline melanoma screening on the Raspberry Pi (needs only: onnxruntime numpy pillow).
# Usage: python pi_infer.py --image lesion.jpg
import argparse, json
import numpy as np
from PIL import Image
import onnxruntime as ort


def heat_colors(x):
    r = np.clip(1.5 - np.abs(4 * x - 3), 0, 1)
    g = np.clip(1.5 - np.abs(4 * x - 2), 0, 1)
    b = np.clip(1.5 - np.abs(4 * x - 1), 0, 1)
    return (np.stack([r, g, b], -1) * 255).astype(np.uint8)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", required=True)
    ap.add_argument("--model", default="mnv3l_int8.onnx")
    ap.add_argument("--head", default="head.npz")
    ap.add_argument("--config", default="config.json")
    ap.add_argument("--out", default="result.png")
    ap.add_argument("--threads", type=int, default=4)
    args = ap.parse_args()

    cfg = json.load(open(args.config))
    mean = np.array(cfg["mean"], dtype=np.float32)
    std = np.array(cfg["std"], dtype=np.float32)

    img = Image.open(args.image).convert("RGB").resize((224, 224), Image.BILINEAR)
    x = ((np.asarray(img, dtype=np.float32) / 255.0 - mean) / std).transpose(2, 0, 1)[None].astype(np.float32)

    so = ort.SessionOptions()
    so.intra_op_num_threads = args.threads
    sess = ort.InferenceSession(args.model, so, providers=["CPUExecutionProvider"])
    logit, feats = sess.run(None, {"image": x})
    prob = float(1 / (1 + np.exp(-logit[0, 0])))

    # heatmap = ReLU(sum_k v_k * A_k): identical to Grad-CAM for this linear head, no gradients needed
    v = np.load(args.head)["w"]
    cam = np.maximum(np.tensordot(v, feats[0], axes=([0], [0])), 0)
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = np.asarray(Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0

    base = np.asarray(img, dtype=np.float32)
    blend = (0.55 * base + 0.45 * heat_colors(cam)).astype(np.uint8)
    Image.fromarray(np.concatenate([base.astype(np.uint8), blend], axis=1)).save(args.out)

    thr = cfg["threshold"]
    verdict = "HIGHER RISK - refer to a clinician" if prob >= thr else "lower risk"
    print(f"Melanoma risk score: {prob:.1%} (threshold {thr:.3f}) -> {verdict}")
    print("Screening aid only; not a diagnosis. Heatmap saved to", args.out)


if __name__ == "__main__":
    main()
'''
(CFG.out_dir / "pi_infer.py").write_text(PI_SCRIPT)
print("Copy these to the Raspberry Pi (64-bit OS):")
for f in ["mnv3l_int8.onnx", "head.npz", "config.json", "pi_infer.py"]:
    print("  ", CFG.out_dir / f, "->", "OK" if (CFG.out_dir / f).exists() else "MISSING")
print("\nOn the Pi:  pip install onnxruntime numpy pillow   then   python pi_infer.py --image lesion.jpg")

## 14. Roadmap and limitations
**Scope:** this notebook produces the trained, quantized and exported model plus the inference script. The Raspberry Pi camera-feed kit
itself (physical device, camera loop, user interface) is a separate, smaller follow-on project that loads `mnv3l_int8.onnx` into an inference loop.

**Next steps**
- Run on real data (for example ISIC) split by lesion/patient, and replace the simulated degradations with a held-out set of real phone photos.
- Evaluate fairness: report sensitivity stratified by skin tone (for example Fitzpatrick type), not just overall.
- Try partially unfreezing the last backbone blocks and compare against the fully frozen backbone.
- Compare against larger or dermatology-specific foundation models server-side as a reference, not as the on-device model.

**Limitations:** the quality-drop curve uses a *simulated* degradation model, the threshold is chosen on a small validation set,
and the numbers are only as trustworthy as the data. This is a research scaffold, not a validated clinical tool.